# Slater determinant preparation with parallel Givens rotations

This standalone example prepares **two particles in four spin-orbitals** ($N=4$, $\eta=2$) using OpenFermion and Cirq. Here $N$ counts spin-orbitals, not the unit cells used elsewhere in AQS. No project imports or external data are needed. Run the cells in order.

## Theory and algorithm

The target state is
$$|\Psi\rangle=b_0^\dagger b_1^\dagger\cdots b_{\eta-1}^\dagger|\mathrm{vac}\rangle,
\qquad b_j^\dagger=\sum_{k=0}^{N-1}Q_{jk}a_k^\dagger.$$
The rows of $Q\in\mathbb C^{\eta\times N}$ are occupied orbitals. The condition $QQ^\dagger=I_\eta$ ensures canonical anticommutation relations for the $b_j$ and a normalized determinant. We sample complex orbitals; this is an arbitrary determinant, not a ground state of a specified SSH Hamiltonian.

A QR-like elimination removes entries of $Q$ with two-mode Givens rotations. Rotations within the occupied orbital space only change the determinant by a global phase. Reversing the elimination prepares the target from the first $\eta$ occupied modes. Disjoint neighboring pairs can be rotated simultaneously on a linear qubit array. This gives quadratic gate count and linear depth scaling, as discussed by [Kivlichan et al., *Phys. Rev. Lett.* **120**, 110501 (2018)](https://arxiv.org/abs/1711.04789). OpenFermion's related implementation is also described by [Jiang et al. (2018)](https://arxiv.org/abs/1711.05395); the literal layer count returned below need not equal an optimized depth bound from another schedule.

The [OpenFermion API](https://quantumai.google/reference/python/openfermion/circuits/slater_determinant_preparation_circuit) accepts $Q$ and returns a list of parallel layers. Each layer contains tuples `(p, q, theta, phi)`: mode indices and angles in radians. The returned layers are already in **preparation order** and should not be reversed again. The prepared state is correct up to a global phase.

## Environment

Use a Python Jupyter kernel with `numpy`, `scipy`, `openfermion`, and `cirq-core`. In a separate environment these can be installed with `python -m pip install numpy scipy openfermion cirq-core jupyterlab`. This notebook does not install or change packages. Cirq is needed for construction and simulation; its availability is reported in Cell 1.


## Cell 1 — Import dependencies


In [1]:
import numpy as np
import scipy
from scipy.stats import unitary_group
import openfermion
from itertools import combinations

try:
    import cirq
except ImportError:
    cirq = None

np.set_printoptions(precision=5, suppress=True)
print(f"NumPy {np.__version__}; SciPy {scipy.__version__}; OpenFermion {openfermion.__version__}")
print(f"Cirq {cirq.__version__}" if cirq is not None else
      "Cirq unavailable: install cirq-core in this kernel's environment for Cells 5–6.")


NumPy 2.2.6; SciPy 1.15.3; OpenFermion 1.8.1
Cirq 1.5.0


## Cell 2 — Sample an orthonormal $2\times4$ orbital matrix

Take the first two rows of a Haar-random unitary. A fixed seed makes the example reproducible within the recorded software environment. Use the conjugate transpose when checking complex row orthonormality.


In [2]:
N, eta = 4, 2
seed = 2026
rng = np.random.default_rng(seed)
Q = unitary_group.rvs(N, random_state=rng)[:eta, :]
gram = Q @ Q.conj().T
assert Q.shape == (eta, N)
np.testing.assert_allclose(gram, np.eye(eta), rtol=0, atol=1e-12)
print("Q =")
print(Q)
print("\nQ Q† =")
print(gram)
print(f"Row orthonormality error: {np.linalg.norm(gram - np.eye(eta)):.3e}")


Q =
[[-0.42838-0.21807j  0.15933+0.2151j  -0.62754+0.10539j -0.20147-0.50178j]
 [ 0.34475-0.25778j -0.11716+0.3744j  -0.44002+0.40976j  0.09719+0.53836j]]

Q Q† =
[[1.+0.j 0.+0.j]
 [0.-0.j 1.+0.j]]
Row orthonormality error: 8.403e-16


## Cell 3 — Obtain the preparation layers


In [3]:
circuit_description = openfermion.circuits.slater_determinant_preparation_circuit(Q)
print(f"Parallel Givens layers: {len(circuit_description)}")
print(f"Two-mode rotations: {sum(len(layer) for layer in circuit_description)}")


Parallel Givens layers: 3
Two-mode rotations: 4


## Cell 4 — Inspect the operations

The checks below verify nearest-neighbor connectivity and ensure no two gates in a layer share a mode. A layer is a parallel scheduling unit, not a claim about device-specific native gate duration.


In [4]:
for layer_index, layer in enumerate(circuit_description, start=1):
    used_modes = set()
    print(f"Layer {layer_index} ({len(layer)} rotation(s)):")
    for operation in layer:
        if not (isinstance(operation, tuple) and len(operation) == 4):
            raise ValueError(f"Unexpected operation: {operation!r}")
        p, q, theta, phi = operation
        assert 0 <= p < q < N and q == p + 1
        assert not used_modes.intersection((p, q)), "Layer is not disjoint"
        used_modes.update((p, q))
        print(f"  modes ({p}, {q}): theta = {theta:+.8f}, phi = {phi:+.8f} rad")


Layer 1 (1 rotation(s)):
  modes (1, 2): theta = +0.98221714, phi = -0.35635571 rad
Layer 2 (2 rotation(s)):
  modes (0, 1): theta = +1.16431248, phi = +0.06778608 rad
  modes (2, 3): theta = +1.18051213, phi = +1.69444608 rad
Layer 3 (1 rotation(s)):
  modes (1, 2): theta = +1.07632431, phi = -1.91826729 rad


## Cell 5 — Construct and display a Cirq circuit

We explicitly order qubits as $(q_0,q_1,q_2,q_3)$ and label kets as $|n_0n_1n_2n_3\rangle$. Thus $X_0X_1|0000\rangle=|1100\rangle$, whose Cirq statevector index is 12. Cirq uses the first qubit in this specified order as the most significant bit.

For operands $(p,q)$, the OpenFermion rotation is represented in Cirq's ordered basis $(|00\rangle,|01\rangle,|10\rangle,|11\rangle)$ by
$$U(\theta,\phi)=\begin{pmatrix}
1&0&0&0\\
0&e^{i\phi}\cos\theta&-e^{i\phi}\sin\theta&0\\
0&\sin\theta&\cos\theta&0\\
0&0&0&e^{i\phi}
\end{pmatrix}.$$
It mixes the single-occupancy states and retains the determinant phase $e^{i\phi}$ on $|11\rangle$. Both the rotation sign and this phase matter for complex orbitals.

This function plays the same bridging role as AQS's `_givens_instruction`, but its matrix uses **Cirq operand ordering**. AQS's Qiskit matrix orders the two operands differently; it cannot be copied here unchanged. Each returned layer becomes one explicit `cirq.Moment` of disjoint two-qubit matrix gates. These are executable simulation gates, not a native hardware decomposition. Diagram labels `G1`, `G2`, … identify layers; angles are printed above.


In [5]:
if cirq is None:
    raise ImportError("Cell 5 requires cirq-core; install it in the active kernel and rerun Cell 1.")

def givens_matrix(theta, phi):
    """OpenFermion preparation rotation in Cirq's |00>, |01>, |10>, |11> basis."""
    c, s = np.cos(theta), np.sin(theta)
    phase = np.exp(1j * phi)
    return np.array([
        [1, 0, 0, 0],
        [0, phase * c, -phase * s, 0],
        [0, s, c, 0],
        [0, 0, 0, phase],
    ], dtype=complex)

qubits = cirq.LineQubit.range(N)
# The simulator starts in |0000>; these X gates prepare |1100> once.
moments = [cirq.Moment(cirq.X(qubits[p]) for p in range(eta))]
for layer_index, layer in enumerate(circuit_description, start=1):
    operations = []
    for p, q, theta, phi in layer:
        matrix = givens_matrix(theta, phi)
        np.testing.assert_allclose(matrix.conj().T @ matrix, np.eye(4), atol=1e-12)
        operations.append(cirq.MatrixGate(matrix, name=f"G{layer_index}").on(qubits[p], qubits[q]))
    moments.append(cirq.Moment(operations))

circuit = cirq.Circuit(moments)
assert len(circuit) == 1 + len(circuit_description)
print(circuit.to_text_diagram(transpose=False))
print(f"\nMoments: {len(circuit)} (one occupation layer plus {len(circuit_description)} Givens layers)")


0: ───X───────────G2[1]───────────
                  │
1: ───X───G1[1]───G2[2]───G3[1]───
          │               │
2: ───────G1[2]───G2[1]───G3[2]───
                  │
3: ───────────────G2[2]───────────

Moments: 4 (one occupation layer plus 3 Givens layers)


## Cell 6 — Verify against the determinant formula

A circuit diagram alone does not establish correct preparation. Independently expand the target in the occupation basis: for ordered occupied modes $S=(s_0<s_1)$, the coefficient is $\det Q_{[:,S]}$. This follows directly from fermionic antisymmetry and does not reuse the Givens decomposition.

Both vectors below use the explicitly specified Cirq qubit order, with mode 0 on the left. No OpenFermion sparse matrix or Qiskit statevector is used. For comparison with AQS/Qiskit, first convert the basis ordering. Check normalization, support only in the two-particle sector, and fidelity with the determinant reference; align a single global phase before checking amplitudes.


In [6]:
simulator = cirq.Simulator(dtype=np.complex128)
psi = simulator.simulate(circuit, qubit_order=qubits, initial_state=0).final_state_vector

reference = np.zeros(2**N, dtype=complex)
for occupied_modes in combinations(range(N), eta):
    # Big-endian index for the displayed ket |n0 n1 n2 n3>.
    index = sum(1 << (N - 1 - mode) for mode in occupied_modes)
    reference[index] = np.linalg.det(Q[:, occupied_modes])

np.testing.assert_allclose(np.linalg.norm(reference), 1.0, atol=1e-12)
np.testing.assert_allclose(np.linalg.norm(psi), 1.0, atol=1e-12)
particle_numbers = np.array([bin(index).count("1") for index in range(2**N)])
probabilities = np.abs(psi)**2
leakage = probabilities[particle_numbers != eta].sum()
mean_particles = probabilities @ particle_numbers
assert leakage < 1e-12
np.testing.assert_allclose(mean_particles, eta, atol=1e-12)

overlap = np.vdot(reference, psi)
fidelity = abs(overlap)**2
np.testing.assert_allclose(fidelity, 1.0, rtol=0, atol=1e-12)
aligned_psi = psi * np.exp(-1j * np.angle(overlap))
np.testing.assert_allclose(aligned_psi, reference, rtol=0, atol=1e-12)
print(f"Fidelity with independent determinant: {fidelity:.12f}")
print(f"Mean particle number: {mean_particles:.12f}")
print(f"Probability outside eta={eta}: {leakage:.3e}")
print("\nKet     Reference amplitude       Prepared amplitude (phase aligned)")
for index in np.flatnonzero(particle_numbers == eta):
    print(f"|{index:0{N}b}>  {reference[index]:+.8f}    {aligned_psi[index]:+.8f}")


Fidelity with independent determinant: 1.000000000000
Mean particle number: 2.000000000000
Probability outside eta=2: 0.000e+00

Ket     Reference amplitude       Prepared amplitude (phase aligned)
|0011>  -0.41198994-0.46583580j    -0.41198994-0.46583580j
|0101>  -0.31178139+0.12332500j    -0.31178139+0.12332500j
|0110>  -0.19230662+0.21793330j    -0.19230662+0.21793330j
|1001>  +0.27457573-0.13076073j    +0.27457573-0.13076073j
|1010>  +0.46702824-0.27767963j    +0.46702824-0.27767963j
|1100>  +0.02145344-0.16791680j    +0.02145344-0.16791680j


## What this demonstrates

OpenFermion computes the rotation schedule from the occupied orbitals. The explicit gate matrices translate that schedule into a Cirq circuit, retaining the parallel layers. Direct determinant amplitudes verify the resulting state up to its physically irrelevant global phase.

The dense statevector and determinant expansion are small-example validation tools; they are not the scalable part of the preparation algorithm.

Validation of this example also reintroduced two convention defects separately: negating every Givens angle and omitting the phase on |11>. Both failed the independent determinant fidelity check. All six code cells were executed in order with the versions recorded in Cell 1.
